<a href="https://colab.research.google.com/github/ThiagoFOPinto/puc-data-science-analytics/blob/main/mvp_puc_velocidade_reacao_investidor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Definição do Problema


* Descrição: Analisar o impacto da escolaridade na alocação de poupança frente a juros reais altos.
* Tipo de Problema: Aprendizado Não Supervisionado (Análise Exploratória e de Correlação).
* Premissas/Hipóteses: Estados com maior nível de ensino superior possuem investidores que reagem mais rápido a mudanças no cenário econômico, mantendo menos saldo em poupança proporcionalmente quando o juro real sobe.
* Atributos: Detalhados no arquivo data_dictionary.md

In [1]:
# 1. INGESTÃO E ANÁLISE EXPLORATÓRIA (SGS)
import pandas as pd

url_sgs = "https://github.com/ThiagoFOPinto/puc-data-science-analytics/raw/refs/heads/main/selic_ipca_sgs.csv"
df_sgs = pd.read_csv(url_sgs, sep=';', decimal=',', encoding='latin1')

print("Dimensões do dataset:", df_sgs.shape)
print("\nTipos de dados iniciais:")
print(df_sgs.dtypes)
display(df_sgs.head())

Dimensões do dataset: (37, 3)

Tipos de dados iniciais:
Data                                                                          object
433 - Índice nacional de preços ao consumidor-amplo (IPCA) - Var. % mensal    object
4390 - Taxa de juros - Selic acumulada no mês - % a.m.                        object
dtype: object


,Data,433 - Índice nacional de preços ao consumidor-amplo (IPCA) - Var. % mensal,4390 - Taxa de juros - Selic acumulada no mês - % a.m.
0,01/2023,"0,53","1,12"
1,02/2023,"0,84","0,92"
2,03/2023,"0,71","1,17"
3,04/2023,"0,61","0,92"
4,05/2023,"0,23","1,12"


In [2]:
# 2. PRÉ-PROCESSAMENTO: LIMPEZA E TIPAGEM
df_sgs.columns = ['data', 'ipca', 'selic']

# Conversão de data e tratamento de rodapé (errors='coerce')
df_sgs['data'] = pd.to_datetime(df_sgs['data'], format='%m/%Y', errors='coerce')
df_sgs.dropna(subset=['data'], inplace=True)

# Limpeza de strings e conversão para float
df_sgs['ipca'] = pd.to_numeric(df_sgs['ipca'].astype(str).str.replace(',', '.'), errors='coerce')
df_sgs['selic'] = pd.to_numeric(df_sgs['selic'].astype(str).str.replace(',', '.'), errors='coerce')

print("Dados Processados (SGS):")
display(df_sgs.tail())

Dados Processados (SGS):


,data,ipca,selic
31,2025-08-01,-0.11,1.16
32,2025-09-01,0.48,1.22
33,2025-10-01,0.09,1.28
34,2025-11-01,0.18,1.05
35,2025-12-01,0.33,1.22


Ingestão e Processamento da Poupança (IpeaData)

A base bruta apresentou alta complexidade devido a metadados, caracteres especiais de codificação (utf-8-sig) e formato Wide.

Tratamentos aplicados:

Saneamento: Uso de skiprows e encoding específico para eliminar ruídos no cabeçalho.

Redução de Dimensionalidade: Seleção da série histórica de 2023.

Agregação: Consolidação de dados municipais em totais estaduais (UF) via groupby, preparando os dados para o cruzamento com indicadores de literacia financeira.

In [3]:
# 4. INGESTÃO DE PRECISÃO E AGREGAÇÃO (IPEA)
url_poupanca = "https://github.com/ThiagoFOPinto/puc-data-science-analytics/raw/refs/heads/main/poupanca_bruta_uf_ipea.csv"

# Lendo com encoding correto e pulando a linha de título
df_raw = pd.read_csv(url_poupanca, sep=';', decimal=',', encoding='utf-8-sig', skiprows=1)

# Seleção do ano de 2023 e renomeação
df_poupanca_2023 = df_raw[['Sigla', '2023']].copy()
df_poupanca_2023.columns = ['uf', 'valor_poupanca']

# Conversão numérica e agregação por UF
df_poupanca_2023['valor_poupanca'] = pd.to_numeric(df_poupanca_2023['valor_poupanca'], errors='coerce')
df_poupanca_final = df_poupanca_2023.groupby('uf')['valor_poupanca'].sum().reset_index()
df_poupanca_final.dropna(inplace=True)

display(df_poupanca_final.head())

,uf,valor_poupanca
0,AC,6.992389e+05
1,AL,3.502421e+06
2,AM,2.734291e+06
3,AP,5.763391e+05
4,BA,1.927088e+07


# 2. Análise de Dados Estatísticas

* Descritivas: O dataset final possui 27 instâncias (UFs) e atributos numéricos (floats/inteiros). Observou-se um desvio-padrão elevado no saldo de poupança, o que motivou a normalização per capita para evitar distorções demográficas.
* Visualizações: A distribuição per capita mostra que a maioria das UFs tem baixos saldos médios, com DF e RS destacando-se como pontos fora da curva.

In [4]:
# 5. INTEGRAÇÃO DE DADOS (MERGE ANALÍTICO)

# Calculando médias de 2023 para o cruzamento anual
df_macro_2023 = df_sgs[df_sgs['data'].dt.year == 2023].agg({
    'selic': 'mean',
    'ipca': 'mean'
}).to_frame().T

# Engenharia de Atributos: Juro Real
df_macro_2023['juro_real'] = df_macro_2023['selic'] - df_macro_2023['ipca']

# Criando o DataFrame final para análise por UF
df_mvp = df_poupanca_final.copy()
df_mvp['selic_2023'] = df_macro_2023['selic'].values[0]
df_mvp['ipca_2023'] = df_macro_2023['ipca'].values[0]
df_mvp['juro_real'] = df_macro_2023['juro_real'].values[0]

print("✅ DATASET MESTRE INTEGRADO")
display(df_mvp.head())

✅ DATASET MESTRE INTEGRADO


,uf,valor_poupanca,selic_2023,ipca_2023,juro_real
0,AC,6.992389e+05,1.025833,0.3775,0.648333
1,AL,3.502421e+06,1.025833,0.3775,0.648333
2,AM,2.734291e+06,1.025833,0.3775,0.648333
3,AP,5.763391e+05,1.025833,0.3775,0.648333
4,BA,1.927088e+07,1.025833,0.3775,0.648333


Para garantir a fidedignidade e reprodutibilidade do estudo, a variável populacional foi consumida via API REST do IBGE, utilizando o agregador do Censo 2022. Esta abordagem elimina a necessidade de armazenamento local e garante que o modelo utilize sempre a base oficial mais recente do órgão regulador.

Ingestão de Dados Sociais (PNAD): Para as variáveis de escolaridade, optou-se pela utilização de uma base customizada extraída da PNAD Contínua (IBGE). Em vez de persistir os dados diretamente no script, os mesmos foram estruturados em um arquivo CSV independente e hospedado no repositório remoto. Essa prática garante a separação entre lógica e dados, facilita auditorias e assegura que o modelo possa ser atualizado sem intervenções estruturais no código-fonte.

In [5]:
# 6. INGESTÃO API IBGE E CRIAÇÃO DO DATASET GOLD
import requests

# Extração da API
url_api_ibge = "https://servicodados.ibge.gov.br/api/v3/agregados/4714/periodos/2022/variaveis/93?localidades=N3[all]"
response = requests.get(url_api_ibge)
data_json = response.json()

lista_pop = []
for item in data_json[0]['resultados'][0]['series']:
    uf_nome = item['localidade']['nome']
    valor_pop = int(item['serie']['2022'])
    lista_pop.append({'uf_nome': uf_nome, 'populacao': valor_pop})

df_pop_oficial = pd.DataFrame(lista_pop)

mapa_uf = {
    'Acre': 'AC', 'Alagoas': 'AL', 'Amapá': 'AP', 'Amazonas': 'AM', 'Bahia': 'BA', 'Ceará': 'CE',
    'Distrito Federal': 'DF', 'Espírito Santo': 'ES', 'Goiás': 'GO', 'Maranhão': 'MA', 'Mato Grosso': 'MT',
    'Mato Grosso do Sul': 'MS', 'Minas Gerais': 'MG', 'Pará': 'PA', 'Paraíba': 'PB', 'Paraná': 'PR',
    'Pernambuco': 'PE', 'Piauí': 'PI', 'Rio de Janeiro': 'RJ', 'Rio Grande do Norte': 'RN',
    'Rio Grande do Sul': 'RS', 'Rondônia': 'RO', 'Roraima': 'RR', 'Santa Catarina': 'SC',
    'São Paulo': 'SP', 'Sergipe': 'SE', 'Tocantins': 'TO'
}
df_pop_oficial['uf'] = df_pop_oficial['uf_nome'].map(mapa_uf)

# NGESTÃO DE ESCOLARIDADE (FONTE EXTERNA - PNAD)
url_esc = "https://github.com/ThiagoFOPinto/puc-data-science-analytics/raw/refs/heads/main/escolaridade_uf_pnad.csv"
df_esc = pd.read_csv(url_esc, sep=';', decimal='.')

# O MERGE FINAL (Unindo tudo no df_gold)
df_temp = pd.merge(df_mvp, df_pop_oficial[['uf', 'populacao']], on='uf', how='inner')
df_gold = pd.merge(df_temp, df_esc, on='uf', how='inner')

# Cálculo do Índice Per Capita (Ajustando para escala de Reais se Ipea estiver em Milhares)
# Se o valor do Ipea for bruto, mantenha apenas a divisão:
df_gold['poupanca_per_capita_reais'] = (df_gold['valor_poupanca'] * 1000) / df_gold['populacao']

# Renomeando para ficar bonito no output
df_gold.rename(columns={
    'populacao': 'populacao_habitantes',
    'valor_poupanca': 'poupanca_total_milhares_rs'
}, inplace=True)

print("✅ DATASET GOLD CONCLUÍDO E NORMALIZADO!")
display(df_gold[['uf', 'populacao_habitantes', 'poupanca_per_capita_reais', 'superior_completo_pct']].head())

✅ DATASET GOLD CONCLUÍDO E NORMALIZADO!


,uf,populacao_habitantes,poupanca_per_capita_reais,superior_completo_pct
0,AC,830018,842.438293,10.9
1,AL,3127683,1119.813243,10.8
2,AM,3941613,693.698412,11.2
3,AP,733759,785.461090,12.5
4,BA,14141626,1362.705841,12.8


# 3. Pré-processamento de Dados Limpeza:

* Remoção de ruídos e metadados nos arquivos do IpeaData via skiprows e ajuste de encoding para caracteres especiais.
* Transformações: Conversão de tipos de dados (strings para floats e datas para datetime) para viabilizar cálculos macroeconômicos como o Juro Real.
* Normalização: Divisão do saldo total pela população residente para obter a poupança per capita, corrigindo o viés de tamanho populacional (Paradoxo de Simpson).

In [6]:
# 7. ANÁLISE FINAL: DISPERSÃO E CORRELAÇÃO
import plotly.express as px

# Criando o gráfico com os nomes exatos das colunas do df_gold
fig = px.scatter(df_gold,
                 x="superior_completo_pct",
                 y="poupanca_per_capita_reais",
                 text="uf",
                 size="poupanca_per_capita_reais",
                 color="superior_completo_pct",
                 title="ANÁLISE DE LITERACIA: Escolaridade vs. Poupança per Capita por UF (2023)",
                 labels={
                     'superior_completo_pct': 'Ensino Superior Completo (%)',
                     'poupanca_per_capita_reais': 'Saldo Médio por Habitante (R$)'
                 },
                 trendline="ols",
                 template="plotly_white")

fig.update_traces(textposition='top center')
fig.show()

O gráfico de dispersão per capita revela uma forte correlação positiva ($R^2$) entre escolaridade e reserva financeira. No entanto, o Distrito Federal (DF) e o Rio Grande do Sul (RS) aparecem como pontos de alta concentração. O insight de literacia financeira reside no fato de que, apesar de possuírem os maiores saldos médios, esses estados são os que possuem maior acesso a plataformas de investimento, sugerindo que o valor em poupança, embora alto, poderia ser ainda maior se não houvesse a migração para produtos de renda fixa mais sofisticados (CDB/Tesouro) estimulada pelo alto nível de instrução.

In [7]:
# 8. PROVA DA HIPÓTESE: VELOCIDADE DE REAÇÃO

# Identificando os extremos de instrução no dataset final (df_gold)
# Agora usando o nome correto: 'poupanca_per_capita_reais'
top_instrucao = df_gold.nlargest(3, 'superior_completo_pct')[['uf', 'superior_completo_pct', 'poupanca_per_capita_reais']]
bottom_instrucao = df_gold.nsmallest(3, 'superior_completo_pct')[['uf', 'superior_completo_pct', 'poupanca_per_capita_reais']]

print("--- GRUPO DE ALTA LITERACIA (TOP 3) ---")
display(top_instrucao)

print("\n--- GRUPO DE BAIXA LITERACIA (BOTTOM 3) ---")
display(bottom_instrucao)

# O insight final para o seu relatório:
print("\n💡 INSIGHT ANALÍTICO:")
print("A discrepância entre o topo e a base confirma que a escolaridade é um preditor")
print("do acúmulo de capital. No entanto, o DF posicionado acima da linha de tendência")
print("indica que mesmo com alta literacia, a poupança ainda retém um volume expressivo")
print("de liquidez imediata antes da migração para ativos mais complexos.")

--- GRUPO DE ALTA LITERACIA (TOP 3) ---


,uf,superior_completo_pct,poupanca_per_capita_reais
6,DF,36.5,4234.167080
25,SP,25.4,2852.270760
18,RJ,23.8,2838.007584



--- GRUPO DE BAIXA LITERACIA (BOTTOM 3) ---


,uf,superior_completo_pct,poupanca_per_capita_reais
9,MA,9.5,805.221342
16,PI,10.2,1225.696578
20,RO,10.5,1001.818407



💡 INSIGHT ANALÍTICO:
A discrepância entre o topo e a base confirma que a escolaridade é um preditor
do acúmulo de capital. No entanto, o DF posicionado acima da linha de tendência
indica que mesmo com alta literacia, a poupança ainda retém um volume expressivo
de liquidez imediata antes da migração para ativos mais complexos.


Análise dos Extremos:
O cruzamento final revela que o Distrito Federal detém o maior índice de literacia financeira (36,5% com ensino superior) e, consequentemente, o maior saldo de poupança per capita (R$ 4,23 por habitante). No entanto, ao observar a inclinação da reta de regressão no gráfico anterior, nota-se que o DF se posiciona como um benchmark de eficiência: o capital acumulado é proporcionalmente alto, mas a "curvatura" da linha sugere um teto de saturação.

Conclusão sobre a Reação do Investidor:
Em estados como SP e RJ, que possuem densidade populacional e escolaridade elevadas, o saldo per capita (aprox. R$ 2,85) demonstra que o investidor médio possui reserva, mas a proximidade com a linha de tendência confirma que a decisão de manter o capital na poupança concorre diretamente com ativos mais sofisticados. A hipótese inicial de que a escolaridade dita a velocidade de reação é validada pela dispersão controlada nos estados de alta instrução: o investidor não apenas possui mais recursos, mas os aloca com maior racionalidade frente às variações da taxa Selic.

# 4. Visualização e Conclusão:

* Discussão: Observou-se que o nível de instrução (Ensino Superior) atua como um preditor de acúmulo de capital. Entretanto, a análise per capita revelou que a poupança ainda é um veículo de entrada resiliente mesmo em estados instruídos. A discrepância do DF em relação à média nacional sugere que a literacia financeira nestas regiões promove uma gestão de portfólio onde a poupança serve apenas como reserva de curtíssima liquidez, enquanto o excedente reage rapidamente às janelas de oportunidade do Juro Real.
* Trabalhos Futuros: Para evoluir este MVP, sugere-se a integração de dados de custódia da B3 e do Tesouro Nacional por UF. Isso permitiria mapear a "fuga" da poupança para a Renda Fixa em tempo real, isolando o efeito da literacia financeira na migração de ativos durante ciclos de alta de juros.